In [ ]:
# In the name of GOD, the Most Gracious, the Most Merciful

In [ ]:
# pip install scikit-learn==1.3.1 # the required version

In [ ]:
# pip install xgboost==2.0.3 # the required version

In [ ]:
# Python == 3.11.4

In [ ]:
import numpy as np
import pandas as pd
import random
import h5py
import joblib
import time
import itertools


import csv
import json
from json import JSONEncoder
from statistics import mode

In [ ]:
############################ A function to train the DQN agent ##################################  
def Defer_model(num_episodes, num_time_steps, Features, CVSS_df, combined_alert_data, investigation_times):
    


    # Initialise an empty dictionary to store metrics
    metrics_data = {}
    size_alert_left = len(Features)
    lower_bound = 0

    # Load classifer models (for Ensemble model)
    rf_clf = joblib.load('RF_whole_model_CIC_mapped.joblib')
    adb_clf = joblib.load('ADB_whole_model_CIC_mapped.joblib')
    xgb_clf = joblib.load('XGB_whole_model_CIC_mapped.joblib')
    nb_clf = joblib.load('NB_whole_model_CIC_mapped.joblib')
    
    for episode in range(num_episodes):

        episode_key = f"episode_{episode + 1}"  # Use a dynamic key for each episode
        
        # Initialise an empty dictionary for the current episode metrics
        episode_metrics = {}

        # Poisson distribution
        # Mean number of alerts per hour (λ)
        lambda_ = 400
        
        # Number of time steps (hours)
        steps = num_time_steps  # 24
        
        # Generate the number of alerts for each time step using Poisson distribution
        alerts_number_all_time_steps = np.random.poisson(lambda_, steps)
        
        
        for time_step in range(num_time_steps):
            
            start_time_1 = time.time()  # Record the start time 

            
            
            time_step_key = f"time_step_{time_step + 1}"  # Use a dynamic key for each time step
            
            # for my clarification
            time_step = time_step 

            # The total time of run so far
            total_time = (episode * num_time_steps) + time_step

            ########## Alert selection for each time_step ######################################################################
            
            # The number of input alerts to SOC based on Poisson distribution  
            alerts_per_step = alerts_number_all_time_steps[time_step]
        
            # Determine how many alerts to select
            alerts_to_select = min(alerts_per_step, size_alert_left)

            

            upper_bound = lower_bound + alerts_to_select
            
            
            # Select the next batch of alerts
            selected_alert_data = Features.iloc[lower_bound : upper_bound]
            
        
            # Select the reprioritisation values of alerts      
            selected_reprioritisation_data = CVSS_df[lower_bound : upper_bound]
            

            selected_combined_alert_data = combined_alert_data[lower_bound : upper_bound]
            
            
            # alerts for each time_step
            size_alert_left = size_alert_left - len(selected_alert_data)
            lower_bound = upper_bound
            
            ###############################################################################################################                              
            ################ Ensemble prioritisation ######################################################################

            X_new = selected_alert_data.to_numpy()
            

            CVSS_label = selected_reprioritisation_data['CVSS_value']            # Can be used for the analyst
            Identifiers = selected_reprioritisation_data['Identifier'].tolist()
            Attack_types = selected_reprioritisation_data[' Label'].tolist()
            
    
            # Make prioritisation using each individual model
            rf_predictions_mapped = rf_clf.predict(X_new)
            adb_predictions_mapped = adb_clf.predict(X_new)
            xgb_predictions_mapped = xgb_clf.predict(X_new)
            nb_predictions_mapped = nb_clf.predict(X_new)
            

            # Map and De-map y values to/from sequential labels
            # Map
            label_mapping = {0: 0, 2: 1, 3: 2, 4: 3}
            #De-map
            reverse_mapping = {v: k for k, v in label_mapping.items()}

            # predictions with real categories
            rf_predictions = np.vectorize(reverse_mapping.get)(rf_predictions_mapped)
            adb_predictions = np.vectorize(reverse_mapping.get)(adb_predictions_mapped)
            xgb_predictions = np.vectorize(reverse_mapping.get)(xgb_predictions_mapped)
            nb_predictions = np.vectorize(reverse_mapping.get)(nb_predictions_mapped)

            
            
            # Combine the four prediction lists into a single list of tuples
            combined_predictions = zip(rf_predictions, adb_predictions, xgb_predictions, nb_predictions)
        
            # Create a list of lists, where each inner list contains the corresponding values from the four lists
            all_predictions = [list(prediction_tuple) for prediction_tuple in combined_predictions]
        
            all_predictions = np.array(all_predictions)
            
            
            modes = []
            for values in all_predictions:
                mode_value = mode(values)
                modes.append(mode_value)
            
        
            # Choose the predictions from all models using majority voting
            ensemble_prioritisation = modes
            
            
            # Ensure ensemble_prioritisation are numpy arrays
            ensemble_prioritisation = np.array(ensemble_prioritisation)
                

            ######### Building the defer values

            # Set confidence level
            confidence_level = 3
            
            # Initialize defer list
            defer = []
            
            # Iterate over each prediction list and the corresponding ensemble value
            for i in range(len(all_predictions)):
                predictions = all_predictions[i]
                ensemble_value = ensemble_prioritisation[i]
                
                # Count how many predictions match the ensemble value
                matching_count = np.sum(predictions == ensemble_value)
                
                # Deferral logic
                if matching_count >= confidence_level:
                    defer.append(0)  # No deferral
                else:
                    defer.append(1)  # Defer
            

            ##############################################################
            ######## Assigning Ensemble Prioritisation to alerts #########

            alert_list =[]
            # Convert predicted_alerts (prioritised alerts) into a list of dictionaries with required features
            for index, value in enumerate(ensemble_prioritisation):
                priority = ''
                if value == 4:
                    priority = 'Critical'
                elif value == 3:
                    priority = 'High'
                elif value == 2:
                    priority = 'Moderate'
                elif value == 1:
                    priority = 'Low'
                else:
                    priority = 'Normal'

                
                alert_id = f'{total_time}_{index+1}'
                

                # Create the dictionary for each alert
                alert_dict = {
                    'alert_id': alert_id,
                    'Identifier': Identifiers[index],
                    'Attack_type': Attack_types[index],
                    'priority_Ensemble': value,     
                    'priority_Analyst': CVSS_label.iloc[index],
                    'revised_priority': 10,
                    'allocated_time': None,
                    'alert_action': None,
                    'alert_defer': defer[index]
                }


                # Append the dictionary to the list
                alert_list.append(alert_dict)
                

            # Here, we include Normal alerts as well. 
            # But, to avoid changing the name of variables and the code we use the name 'alert_list_without_Normal'
            alert_list_without_Normal = alert_list

                
                
################# Interacting with analyst ########################   

            # lists to collect alerts whose 'revised_priority' are assigned and are not assigned
            alert_list_without_Normal_with_revised_priority = []
            alert_list_without_Normal_without_revised_priority = []

            for alert in alert_list_without_Normal:
                if alert['revised_priority'] != 10:
                    alert_list_without_Normal_with_revised_priority.append(alert)
                else:
                    alert_list_without_Normal_without_revised_priority.append(alert)


            ## We do not sort alerts based on their Ensemble priority to present to DQN, because this is a part of contribution of L2DHF and L2D models do not generally do this
            ## We just keep the name 'sorted_alert_list_without_Normal_without_revised_priority' to be consistent with L2DHF code and do not require to change the rest of code   
            sorted_alert_list_without_Normal_without_revised_priority = alert_list_without_Normal_without_revised_priority
            
            step_length = len(sorted_alert_list_without_Normal_without_revised_priority)
            analyst_time_percentage_for_analysis = 0.8
            remaining_time = int(60 * 60 * analyst_time_percentage_for_analysis)
            

            # Lists of performance
            alerts_after_action_015 = []
            alerts_after_action_01 = [] # this gathers alerts after DQN decision (0: accept, 1: defer)
            

            def calculate_average(list_A):
                if len(list_A) == 0:
                    return 0
                return sum(list_A) / len(list_A)
            
            while remaining_time > 0:
                for index, alert_DQN in enumerate(sorted_alert_list_without_Normal_without_revised_priority):
                    
                    # get investigation time
                    Ensemble_priority = alert_DQN['priority_Ensemble']
                    
                    investigation_time_alert = investigation_times.get(Ensemble_priority)

                    if remaining_time < investigation_time_alert:
                        action = 5  # It means that alert is not presented because of analyst time constraint 
                        alert_DQN['alert_action'] = action
                        alert_DQN['revised_priority'] = alert_DQN['priority_Ensemble']
                        alert_DQN['allocated_time'] = 0
                       
                    else: # Defer
                        
                        if alert_DQN['alert_defer'] == 1: 
                            alert_DQN['alert_action'] = 1
                            alert_DQN['revised_priority'] = alert_DQN['priority_Analyst']  # this is the presentation to the analyst
                            # update remaining time
                            alert_DQN['allocated_time'] = investigation_time_alert
                            remaining_time -= alert_DQN['allocated_time']
                        else:
                            alert_DQN['alert_action'] = 0
                            alert_DQN['revised_priority'] = alert_DQN['priority_Ensemble']
                            # update remaining time
                            alert_DQN['allocated_time'] = 0         
                        
                        
                        alerts_after_action_01.append(alert_DQN)


                if remaining_time > 0:
                    print(f"Loop terminated with remaining time: {remaining_time}")
                    break

                        
            end_time_1 = time.time()  # Record the end time 
            time_step_execution_time = end_time_1 - start_time_1  # Calculate the duration of this loop         
                    
            
            
                    
            # all alerts without Normal of the time step after actions
            Final_alert_list_without_Normal_addressed = list(itertools.chain(alert_list_without_Normal_with_revised_priority, alerts_after_action_01))
            

            # Extract alert_ids from alerts_after_action_01
            alert_ids_after_action_01 = {alert['alert_id'] for alert in alerts_after_action_01}
            
            
            # Un_investigated alerts by analyst
            Final_alert_list_without_Normal_NotAddressed = [
                alert for alert in alert_list_without_Normal_without_revised_priority
                if alert['alert_id'] not in alert_ids_after_action_01
            ]
            
            Analyst_Workload = len(Final_alert_list_without_Normal_NotAddressed)
            
            # Store metrics in the current time step
            current_metrics = {
                "Number_total_alerts": len(selected_alert_data),
                "alert_list_without_Normal": alert_list_without_Normal,
                "alert_list_without_Normal_with_revised_priority": alert_list_without_Normal_with_revised_priority, 
                "alert_list_without_Normal_without_revised_priority": alert_list_without_Normal_without_revised_priority, 
                "alerts_after_action_01": alerts_after_action_01,
                "Final_alert_list_without_Normal_addressed": Final_alert_list_without_Normal_addressed,
                "Final_alert_list_without_Normal_NotAddressed": Final_alert_list_without_Normal_NotAddressed,
                "time_step_execution_time": time_step_execution_time,
                "Analyst_Workload": Analyst_Workload,
                "Remaining_time": remaining_time
                
                
            }
            
            episode_metrics[time_step_key] = current_metrics
            
            class NumpyEncoder(json.JSONEncoder):
                def default(self, obj):
                    if isinstance(obj, np.ndarray):
                        return obj.tolist()
                    elif isinstance(obj, np.integer):
                        return int(obj)
                    elif isinstance(obj, np.floating):
                        return float(obj)
                    return super(NumpyEncoder, self).default(obj)

                
            output_folder = 'C:\\...\\L2D_CICIDS2017\\'

            
            
            # Save the dictionary of metrics to a JSON file
            with open(f'{output_folder}L2D_CICIDS2017_time_step_{total_time + 1}.json', 'w') as outfile:
                json.dump(current_metrics, outfile, cls=NumpyEncoder)

            
            

        # Store the episode metrics in the main dictionary
        metrics_data[episode_key] = episode_metrics
        
        
    return metrics_data

In [ ]:
############################################### Load the alert dataset ######################################################################
################ pca_12_one_hot
#### subset 1, 2, 3, 4, 5 
df_PCA_12_combined_data_CIC_reduced_normalised_1 = pd.read_csv(r'C:\...\df_PCA_12_combined_data_CIC_reduced_normalised_1.csv')
df_PCA_12_combined_data_CIC_reduced_normalised_2 = pd.read_csv(r'C:\...\df_PCA_12_combined_data_CIC_reduced_normalised_2.csv')
df_PCA_12_combined_data_CIC_reduced_normalised_3 = pd.read_csv(r'C:\...\df_PCA_12_combined_data_CIC_reduced_normalised_3.csv')
df_PCA_12_combined_data_CIC_reduced_normalised_4 = pd.read_csv(r'C:\...\df_PCA_12_combined_data_CIC_reduced_normalised_4.csv')
df_PCA_12_combined_data_CIC_reduced_normalised_5 = pd.read_csv(r'C:\...\df_PCA_12_combined_data_CIC_reduced_normalised_5.csv')

In [ ]:
# Shuffle data
df_PCA_12_combined_data_CIC_reduced_normalised_1 = df_PCA_12_combined_data_CIC_reduced_normalised_1.sample(frac=1, random_state=42).reset_index(drop=True)
df_PCA_12_combined_data_CIC_reduced_normalised_2 = df_PCA_12_combined_data_CIC_reduced_normalised_2.sample(frac=1, random_state=42).reset_index(drop=True)
df_PCA_12_combined_data_CIC_reduced_normalised_3 = df_PCA_12_combined_data_CIC_reduced_normalised_3.sample(frac=1, random_state=42).reset_index(drop=True)
df_PCA_12_combined_data_CIC_reduced_normalised_4 = df_PCA_12_combined_data_CIC_reduced_normalised_4.sample(frac=1, random_state=42).reset_index(drop=True)
df_PCA_12_combined_data_CIC_reduced_normalised_5 = df_PCA_12_combined_data_CIC_reduced_normalised_5.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
# Combine 5 sub_datasets to a dataset
combined_alert_data = pd.concat([df_PCA_12_combined_data_CIC_reduced_normalised_1, df_PCA_12_combined_data_CIC_reduced_normalised_2, df_PCA_12_combined_data_CIC_reduced_normalised_3, df_PCA_12_combined_data_CIC_reduced_normalised_4, df_PCA_12_combined_data_CIC_reduced_normalised_5], ignore_index=True)

In [ ]:
# Shuffle combined_alert_data (second shuffle)
combined_alert_data = combined_alert_data.sample(frac=1, random_state=42).reset_index(drop=True)


In [ ]:
df_PCA_12_combined_data_CIC_reduced_normalised_1

In [ ]:
df_PCA_12_combined_data_CIC_reduced_normalised_2

In [ ]:
df_PCA_12_combined_data_CIC_reduced_normalised_3

In [ ]:
df_PCA_12_combined_data_CIC_reduced_normalised_4

In [ ]:
df_PCA_12_combined_data_CIC_reduced_normalised_5

In [ ]:
combined_alert_data

In [ ]:
############# Building the AVAR Storages of categories #################
# We do this for the L2D model to make sure that the Ensemble model recieves the same alerts as other models for comparision purposes
# That is, we remove the alerts we use for storages in other models. We do it here as well 
# Otherwise, the L2D model does not need the storages processes 

In [ ]:
# Number of CVSS categories in the dataset
cvss_counts = combined_alert_data['CVSS'].value_counts()
cvss_counts

In [ ]:
# Building the AVAR storages of Critical, High, Medium, Low, and Normal
# Set the desired sample sizes
sample_sizes = {
    'Critical': 5,
    'High': 5,
    'Moderate': 5,
    'Low': 0,
    'Normal': 5
}

# Initialise a dictionary to hold the sampled df
sampled_dfs = {}

# Initialise a list to collect the indices of sampled rows
sampled_indices = []

# Iterate over the sample sizes dictionary
for cvss_value, sample_size in sample_sizes.items():
    # Filter the df for the current CVSS value
    filtered_combined_alert_data = combined_alert_data[combined_alert_data['CVSS'] == cvss_value]
    
    # Randomly sample the specified number of rows
    sampled_df = filtered_combined_alert_data.sample(n=sample_size, random_state=42)
    
    
    sampled_dfs[cvss_value] = sampled_df
    
    
    sampled_indices.extend(sampled_df.index)

# Drop the sampled rows from combined_alert_data
combined_alert_data = combined_alert_data.drop(sampled_indices)

# Separate sets for each CVSS value 
critical_set = sampled_dfs['Critical']
high_set = sampled_dfs['High']
moderate_set = sampled_dfs['Moderate']
low_set = sampled_dfs['Low']
normal_set = sampled_dfs['Normal']

In [ ]:
# Save storages to CSV files
critical_set.to_csv('critical_set.csv', index=False)
high_set.to_csv('high_set.csv', index=False)
moderate_set.to_csv('moderate_set.csv', index=False)
#low_set.to_csv('low_set.csv', index=False) # Low category is removed because it does not exist in CICIDS2017 dataset
normal_set.to_csv('normal_set.csv', index=False)

In [ ]:
critical_set

In [ ]:
high_set

In [ ]:
moderate_set

In [ ]:
low_set

In [ ]:
normal_set

In [ ]:
# Remove labels form storages
columns_to_remove = [' Label', 'CVSS', 'CVSS_value', 'Identifier']
Critical_storage = critical_set.drop(columns=columns_to_remove, axis=1)
High_storage = high_set.drop(columns=columns_to_remove, axis=1)
Moderate_storage = moderate_set.drop(columns=columns_to_remove, axis=1) 
Low_storage = low_set.drop(columns=columns_to_remove, axis=1)
Normal_storage = normal_set.drop(columns=columns_to_remove, axis=1)

In [ ]:
print('Len Critical_storage =',len(Critical_storage))
print('Len High_storage =', len(High_storage))
print('Len Moderate_storage =', len(Moderate_storage))
print('Len Low_storage =', len(Low_storage))
print('Len Normal_storage =', len(Normal_storage))

In [ ]:
Critical_storage

In [ ]:
High_storage

In [ ]:
Moderate_storage

In [ ]:
Low_storage

In [ ]:
Normal_storage

In [ ]:
######### Building alerts without labels for prioritisation and separate labels ############

In [ ]:
# Separating alert features and re-prioritisation columns 
columns_to_remove = [' Label', 'CVSS', 'CVSS_value', 'Identifier']
Features = combined_alert_data.drop(columns=columns_to_remove, axis=1)
CVSS_df = combined_alert_data[['CVSS_value', ' Label', 'Identifier']]

In [ ]:
CVSS_df

In [ ]:
Features

In [ ]:
############################################## Main program ###########################################
if __name__ == "__main__":
      
    # time parameters
    num_episodes = 7 * 12  # days of SOC working 12*7=84
    num_time_steps = 24  # hours of day for SOC working, # one working shift  
    
    # investigation times of analyst for alert categories (in seconds)
    investigation_times = {
        4: 270,   
        3: 210,   
        2: 120,   
        1: 90,    
        0: 60    
    }
    
       
    # Run the model
        
    metrics_data = Defer_model(num_episodes, num_time_steps, Features, CVSS_df, combined_alert_data, investigation_times)
    